In [ ]:
import xml.sax
from collections import defaultdict

KEEP_LANG = {"en", "da"}

concepts = defaultdict(lambda: {
    "en": None,
    "da": None,
    "broader": [],
    "narrower": []
})

current_concept = None
current_lang = None
collect_text = False
buffer = []

class ExtractHandler(xml.sax.ContentHandler):
    def startElement(self, name, attrs):
        global current_concept, current_lang, collect_text

        # Track concept
        if name == "skos:Concept":
            current_concept = attrs.get("rdf:about")

        # Track broader/narrower relations
        if name == "skos:broader":
            uri = attrs.get("rdf:resource")
            if current_concept and uri:
                concepts[current_concept]["broader"].append(uri)

        if name == "skos:narrower":
            uri = attrs.get("rdf:resource")
            if current_concept and uri:
                concepts[current_concept]["narrower"].append(uri)

        # Detect label-bearing element (SKOS or SKOS-XL literalForm)
        lang = attrs.get("xml:lang")
        if lang in KEEP_LANG:
            current_lang = lang
            collect_text = True
            buffer.clear()

    def characters(self, content):
        if collect_text:
            buffer.append(content)

    def endElement(self, name):
        global current_concept, current_lang, collect_text

        # Store collected label
        if collect_text and name.endswith("prefLabel") or name.endswith("literalForm"):
            text = "".join(buffer).strip()
            if current_concept and current_lang and text:
                concepts[current_concept][current_lang] = text

            collect_text = False
            current_lang = None
            buffer.clear()

        # End of concept
        if name == "skos:Concept":
            current_concept = None

parser = xml.sax.make_parser()
parser.setContentHandler(ExtractHandler())
parser.parse("esco-v1.2.0.rdf")

print("Concepts extracted:", len(concepts))

In [ ]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import SKOS, RDF

g = Graph()

for uri, data in tqdm(concepts.items()):
    s = URIRef(uri)

    # Type
    g.add((s, RDF.type, SKOS.Concept))

    # Add labels for this concept
    if data["en"]:
        g.add((s, SKOS.prefLabel, Literal(data["en"], lang="en")))
    if data["da"]:
        g.add((s, SKOS.prefLabel, Literal(data["da"], lang="da")))

    # Hierarchy with embedded labels
    for b in data["broader"]:
        b_uri = URIRef(b)
        g.add((s, SKOS.broader, b_uri))

        # Add labels for the broader node itself
        b_data = concepts.get(b, {})
        if b_data.get("en"):
            g.add((b_uri, SKOS.prefLabel, Literal(b_data["en"], lang="en")))
        if b_data.get("da"):
            g.add((b_uri, SKOS.prefLabel, Literal(b_data["da"], lang="da")))

    # (Optional: also embed narrower, but broader is enough)
    
g.serialize("esco_cleaned.rdf", format="pretty-xml")

In [ ]:
skill_prefix = "http://data.europa.eu/esco/skill/"

all_skills = {
    str(s)
    for s in g.subjects()
    if str(s).startswith(skill_prefix)
}

In [ ]:
edges = {}

for s in all_skills:
    broader_parents = [
        str(o)
        for o in g.objects(s, SKOS.broader)
        if str(o).startswith(skill_prefix)
    ]
    edges[s] = broader_parents

In [ ]:
root_skills = [s for s, parents in edges.items() if len(parents) == 0]

In [ ]:
from rdflib.namespace import SKOS
from rdflib import Literal, URIRef

def get_en_label(uri):
    for lbl in g.objects(URIRef(uri), SKOS.prefLabel):
        if isinstance(lbl, Literal) and lbl.language == 'en':
            return str(lbl)
    return None

# roots from your edges dict
root_skills = [s for s, parents in edges.items() if len(parents) == 0]

root_names = [get_en_label(s) for s in root_skills]
root_names = [name for name in root_names if name]  # drop None

len(root_names)

In [ ]:
def print_tree(node, prefix=""):
    label = G.nodes[node]["en"] or "(no label)"
    
    # Print this node
    print(prefix + label)

    # Collect children (narrower concepts)
    children = list(G.successors(node))
    children.sort(key=lambda c: G.nodes[c]["en"] or "")

    for i, child in enumerate(children):
        is_last = (i == len(children) - 1)
        branch = "└── " if is_last else "├── "
        extension = "    " if is_last else "│   "
        print_tree(child, prefix + branch)

# Print only one top-level tree first
print_tree(roots[0])